<img src="https://cdn.jsdelivr.net/gh/aureliensalas/python_finance@main/ressources/img/logo_macmia.png" alt="Banque des Territoires · France 2030 · MACMIA" width="520">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aureliensalas/python_finance/blob/main/ml_crypto/assignment/fiche_desk.ipynb)

# La fiche de desk

**Travail en autonomie** · bloc 4 · à faire après la séance 7

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.

Ce notebook n'est pas noté. Il sert à vérifier, seul, que vous savez faire ce que la séance 7 a montré — et à repartir avec un document qui a de l'allure.

## La commande

Premier jour sur le **desk crypto** d'une société de gestion. Vous avez à peine posé votre café que trois collègues passent à votre bureau. Tous parlent de la même monnaie.

> 🧑‍💼 **Karim, le gérant** : « J'ai une grosse position sur cette monnaie. Je crois en *elle*, pas au marché crypto en général. Je peux garder l'une sans l'autre ? »
>
> 👩‍💻 **Léa, responsable de l'exécution** : « On doit en vendre beaucoup la semaine prochaine. Combien va-t-il s'en échanger, par jour, sur les cinq prochains jours ? »
>
> 🧮 **Tom, le risk manager** : « Je fixe les limites de risque ce soir. Quelle agitation demain ? »

Vous allez produire **une seule fiche** qui répond aux trois. C'est ce qu'un desk affiche le matin.

**Prenez position maintenant, avant de calculer.** Notez vos deux réponses quelque part, on y revient à la fin :

1. Si Karim se couvre contre le marché, quelle part de son risque va disparaître : **10 %, 30 %, 50 % ou 70 %** ?
2. Entre le volume des cinq prochains jours (Léa) et l'agitation de demain (Tom), **lequel sera le plus facile à prévoir** ?

## Comment on va procéder

La **mise en page est déjà écrite** : une fonction vous est fournie, elle dessine la fiche. Votre travail est de calculer ce qui va dedans — et la séance 7 vous a tout appris, à deux gestes près, qu'on vous montrera sur place.

1. on appelle la fonction tout de suite : la fiche apparaît **vide** ;
2. à chaque étape, vous calculez un morceau, **sous les noms de variables demandés** ;
3. on rappelle `afficher_desk(MONNAIE)` : la fonction va chercher toute seule vos variables, et la fiche se remplit.

Chaque étape a la même forme : **ce qu'on cherche**, **où c'était** dans le cours, **le piège** à éviter, une cellule vide pour vous, et une cellule `verifier` qui dit `OK` ou `A REVOIR` avec un indice.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

pd.set_option("display.max_rows", 12)
BASE = "https://cdn.jsdelivr.net/gh/aureliensalas/python_finance@main/pandas_crypto/data/"        # crypto.csv
BASE_STATS = "https://cdn.jsdelivr.net/gh/aureliensalas/python_finance@main/stats_crypto/data/"   # rendements.csv
BASE_ML = "https://cdn.jsdelivr.net/gh/aureliensalas/python_finance@main/ml_crypto/data/"         # volatilite.csv, minutes_10oct2025.csv


def verifier(nom, condition, indice=""):
    if condition:
        print("OK       -", nom)
    else:
        print("A REVOIR -", nom, ":", indice)

La cellule suivante contient la fonction qui dessine la fiche. Elle est longue, et **vous n'avez pas à la lire** : exécutez-la, puis oubliez-la.

In [ ]:
ENCRE, GRIS, BLEU, ROUGE, VERT, ORANGE, FOND = "#1F2933", "#9AA5B1", "#1F4E79", "#B4341F", "#2E7D5B", "#B26B00", "#F5F7F9"
HALO = dict(facecolor="white", alpha=0.92, edgecolor="none", pad=2.2)
_vide = lambda v: v is None or (isinstance(v, float) and np.isnan(v))


def _attente(a, titre):
    a.set_xticks([]); a.set_yticks([])
    for s in ("top", "right", "bottom", "left"): a.spines[s].set_visible(False)
    a.add_patch(plt.Rectangle((0, 0), 1, 1, transform=a.transAxes, facecolor=FOND,
                              edgecolor="#DCE2E8", lw=1, zorder=0))
    a.set_title(titre, loc="left", fontsize=10.5, color=GRIS, fontweight="bold")
    a.text(0.5, 0.5, "à compléter", transform=a.transAxes, fontsize=12, color=GRIS,
           ha="center", va="center", style="italic")


def _habiller(a, titre):
    a.set_title(titre, loc="left", fontsize=10.5, color=ENCRE, fontweight="bold")
    for s in ("top", "right"): a.spines[s].set_visible(False)
    a.tick_params(labelsize=7.5, colors=GRIS)


def _tuile(fig, case, titre, val, fmt, sous, coul=ENCRE):
    t = fig.add_subplot(case); t.axis("off")
    t.add_patch(plt.Rectangle((0, 0), 1, 1, transform=t.transAxes, color=FOND, zorder=0))
    t.add_patch(plt.Rectangle((0, 0.92), 1, 0.08, transform=t.transAxes,
                              color=GRIS if _vide(val) else BLEU, zorder=1))
    t.text(0.5, 0.72, titre, transform=t.transAxes, fontsize=6.8, color=GRIS, ha="center", fontweight="bold")
    if _vide(val):
        t.text(0.5, 0.36, "à compléter", transform=t.transAxes, fontsize=9.5, color=GRIS, ha="center", style="italic")
    else:
        t.text(0.5, 0.38, fmt % val if isinstance(fmt, str) else fmt(val), transform=t.transAxes,
               fontsize=14, color=coul, ha="center", fontweight="bold")
        t.text(0.5, 0.10, sous, transform=t.transAxes, fontsize=6.8, color=GRIS, ha="center")


def _section(fig, case, texte, qui):
    a = fig.add_subplot(case); a.axis("off")
    a.plot([0, 1], [0.35, 0.35], transform=a.transAxes, color="#DCE2E8", lw=1.2, zorder=0)
    a.text(0.0, 0.35, "  " + texte + "  ", transform=a.transAxes, fontsize=11, color=BLEU,
           va="center", fontweight="bold", bbox=dict(facecolor="white", edgecolor="none", pad=1))
    a.text(1.0, 0.35, "  " + qui, transform=a.transAxes, fontsize=8.5, color=GRIS, ha="right",
           va="center", style="italic", bbox=dict(facecolor="white", edgecolor="none", pad=1))


def _tableau(a, titre, mesures, unite):
    """Trois modèles × trois mesures ; le meilleur de chaque colonne en gras vert."""
    if mesures is None:
        _attente(a, titre); return
    a.axis("off")
    a.set_title(titre, loc="left", fontsize=10.5, color=ENCRE, fontweight="bold")
    cols = list(mesures.columns)
    a.text(0.02, 0.86, "", transform=a.transAxes)
    for j, c in enumerate(cols):
        a.text(0.50 + 0.17 * j, 0.84, c + (f"\n({unite})" if c != "R²" else "\n"), transform=a.transAxes,
               fontsize=8, color=GRIS, ha="center", va="center", fontweight="bold")
    for i, nom in enumerate(mesures.index):
        y = 0.56 - 0.22 * i
        a.add_patch(plt.Rectangle((0, y - 0.10), 1, 0.20, transform=a.transAxes,
                                  color=FOND if i % 2 == 0 else "white", zorder=0))
        a.text(0.02, y, nom, transform=a.transAxes, fontsize=8.5, color=ENCRE, va="center")
        for j, c in enumerate(cols):
            v = mesures.loc[nom, c]
            meilleur = (v == mesures[c].max()) if c == "R²" else (v == mesures[c].min())
            a.text(0.50 + 0.17 * j, y, ("%+.2f" if c == "R²" else "%.2f") % v, transform=a.transAxes,
                   fontsize=9.5, ha="center", va="center", color=VERT if meilleur else ENCRE,
                   fontweight="bold" if meilleur else "normal")


def dessiner_desk(monnaie, periode=None,
                  r=None, beta=None, p_beta=None, nu=None, couvert=None,
                  vol_nue=None, vol_couverte=None, reduction=None, pire_nu=None, pire_couvert=None,
                  dates_v=None, reel_v=None, prevu_v=None, mesures_v=None, prevision_5j=None,
                  minutes_jour=None, rv_10oct=None, rendement_10oct=None, rv_normal=None,
                  dates_h=None, reel_h=None, prevu_h=None, mesures_h=None, prevision_demain=None,
                  rapport=None, part_pire=None, pire_date=None):

    fig = plt.figure(figsize=(11.5, 19.5), facecolor="white")
    gs = GridSpec(10, 5, figure=fig,
                  height_ratios=[0.48, 0.30, 1.45, 0.72, 0.30, 1.45, 0.30, 1.30, 1.45, 0.72],
                  hspace=0.62, wspace=0.34, left=0.06, right=0.965, top=0.978, bottom=0.03)

    # ---------------------------------------------------------------- bandeau
    a = fig.add_subplot(gs[0, :]); a.axis("off")
    a.add_patch(plt.Rectangle((0, 0), 1, 1, transform=a.transAxes, color=BLEU, zorder=0))
    a.text(0.022, 0.60, monnaie, transform=a.transAxes, fontsize=24, color="white", fontweight="bold", va="center")
    a.text(0.022, 0.18, periode or "", transform=a.transAxes, fontsize=8.5, color="#C3D0DC", va="center")
    a.text(0.978, 0.50, "FICHE DE DESK", transform=a.transAxes, fontsize=11, color="white",
           ha="right", va="center", fontweight="bold", alpha=0.9)

    # ---------------------------------------------------------------- 1. couverture
    _section(fig, gs[1, :], "1 · COUVERTURE", "le gérant : « garder la monnaie, pas le marché »")
    a = fig.add_subplot(gs[2, :])
    if nu is None or couvert is None or r is None:
        _attente(a, "10 000 € sur la période de test : position nue et position couverte")
    else:
        d = r["date"][nu.index]
        a.plot(d, 10000 * (1 + nu / 100).cumprod(), color=GRIS, lw=1.3, label=f"{monnaie} seul")
        a.plot(d, 10000 * (1 + couvert / 100).cumprod(), color=BLEU, lw=1.8, label=f"{monnaie} couvert par du BTC")
        a.axhline(10000, color=ENCRE, lw=0.6, ls=":")
        a.legend(loc="upper left", frameon=False, fontsize=8.5)
        a.set_ylabel("euros", fontsize=8)
        _habiller(a, "10 000 € sur la période de test : position nue et position couverte")
    cases = [gs[3, k] for k in range(5)]
    _tuile(fig, cases[0], "BÊTA", beta, "%.2f", "appris sur l'entraînement")
    _tuile(fig, cases[1], "P-VALUE DU BÊTA", p_beta, lambda v: "< 0,001" if v < 0.001 else ("%.3f" % v).replace(".", ","),
           "ce n'est pas le hasard", VERT)
    _tuile(fig, cases[2], "VOLATILITÉ", None if _vide(vol_nue) or _vide(vol_couverte) else vol_couverte,
           lambda v: ("%.2f → %.2f %%" % (vol_nue, v)).replace(".", ","), "nue → couverte, par jour")
    _tuile(fig, cases[3], "RISQUE SUPPRIMÉ", reduction, lambda v: ("−%.0f %%" % v), "de la volatilité", VERT)
    _tuile(fig, cases[4], "PIRE JOUR", None if _vide(pire_nu) or _vide(pire_couvert) else pire_couvert,
           lambda v: ("%.1f → %.1f %%" % (pire_nu, v)).replace(".", ","), "nu → couvert")

    # ---------------------------------------------------------------- 2. liquidité
    _section(fig, gs[4, :], "2 · LIQUIDITÉ", "l'exécution : « combien va-t-il s'échanger cette semaine ? »")
    a = fig.add_subplot(gs[5, :3])
    if reel_v is None or prevu_v is None or dates_v is None:
        _attente(a, "Volume moyen des 5 jours suivants : réel et prévu")
    else:
        a.plot(dates_v, reel_v, color=GRIS, lw=1.1, label="réel")
        a.plot(dates_v, prevu_v, color=BLEU, lw=1.5, label="prévu la veille")
        a.legend(loc="upper left", frameon=False, fontsize=8)
        a.set_ylabel("milliards de $ par jour", fontsize=8)
        _habiller(a, "Volume moyen des 5 jours suivants : réel et prévu")
    _tableau(fig.add_subplot(gs[5, 3:]), "Trois prévisions, sur le test", mesures_v, "Md$")

    # ---------------------------------------------------------------- 3. volatilité
    _section(fig, gs[6, :], "3 · VOLATILITÉ", "le risk manager : « quelle agitation demain ? »")
    a = fig.add_subplot(gs[7, :2])
    if minutes_jour is None:
        _attente(a, "Le 10 octobre 2025, minute par minute")
    else:
        a.plot(range(len(minutes_jour)), minutes_jour["close"].values, color=ROUGE, lw=1.0)
        a.set_xticks([0, 360, 720, 1080, 1439]); a.set_xticklabels(["0 h", "6 h", "12 h", "18 h", "24 h"])
        a.set_ylabel("dollars", fontsize=8)
        _habiller(a, "Le 10 octobre 2025, minute par minute")
        if not _vide(rv_10oct):
            txt = f"volatilité réalisée : {rv_10oct:.1f} %".replace(".", ",")
            if not _vide(rv_normal):
                txt += f", {rv_10oct / rv_normal:.0f} fois une journée ordinaire".replace(".", ",")
            if not _vide(rendement_10oct):
                txt += f"\nrendement de la journée : {rendement_10oct:+.1f} %".replace(".", ",")
            a.text(0.02, 0.06, txt, transform=a.transAxes, fontsize=8.5, color=ENCRE, bbox=HALO)
    _tableau(fig.add_subplot(gs[7, 2:]), "Trois prévisions de la volatilité de demain, sur le test", mesures_h, "pts")
    a = fig.add_subplot(gs[8, :])
    if reel_h is None or prevu_h is None or dates_h is None:
        _attente(a, "Volatilité réalisée, jour par jour : réelle et prévue la veille")
    else:
        dh = pd.Series(pd.to_datetime(np.asarray(dates_h))) + pd.Timedelta(days=1)   # le jour prévu
        a.plot(dh, np.asarray(reel_h), color=GRIS, lw=1.0, label="réelle")
        a.plot(dh, np.asarray(prevu_h), color=BLEU, lw=1.5, label="prévue la veille (HAR)")
        if pire_date is not None:
            k = int(np.argmax(np.abs(np.asarray(reel_h) - np.asarray(prevu_h))))
            a.scatter([dh[k]], [np.asarray(reel_h)[k]], s=70, color=ROUGE, zorder=5)
            a.annotate(f"  {dh[k]:%d/%m/%Y} : prévu {np.asarray(prevu_h)[k]:.1f}, réel {np.asarray(reel_h)[k]:.1f}".replace(".", ","),
                       (dh[k], np.asarray(reel_h)[k]), fontsize=8.5, color=ROUGE, va="top", fontweight="bold")
        a.legend(loc="upper right", frameon=False, fontsize=8)
        a.set_ylabel("% par jour", fontsize=8)
        _habiller(a, "Volatilité réalisée, jour par jour : réelle et prévue la veille")
    cases = [gs[9, k] for k in range(5)]
    _tuile(fig, cases[0], "PRÉVISION : 5 PROCHAINS JOURS", prevision_5j, lambda v: ("%.1f Md$" % v).replace(".", ","), "de volume par jour")
    _tuile(fig, cases[1], "PRÉVISION : VOLATILITÉ DEMAIN", prevision_demain, lambda v: ("%.2f %%" % v).replace(".", ","), "à donner au risk manager")
    _tuile(fig, cases[2], "RMSE / MAE", rapport, lambda v: ("%.2f" % v).replace(".", ","),
           "au-dessus de 2 : une journée domine", ORANGE if not _vide(rapport) and rapport > 2 else ENCRE)
    _tuile(fig, cases[3], "POIDS DE LA PIRE JOURNÉE", part_pire, lambda v: "%.0f %%" % v, "dans l'erreur au carré",
           ORANGE if not _vide(part_pire) and part_pire > 40 else ENCRE)
    _tuile(fig, cases[4], "LA PIRE PRÉVISION", None if pire_date is None else 1.0,
           lambda v: f"{pd.Timestamp(pire_date) + pd.Timedelta(days=1):%d/%m/%Y}", "le jour prévu, raté par le modèle", ROUGE)


def _mesures(*noms):
    """Assemble le tableau 3 × 3 à partir des neuf variables de l'étudiant, s'il les a toutes."""
    g = globals()
    if not all(n in g for n in noms):
        return None
    lignes = [[g[noms[3 * i]], g[noms[3 * i + 1]], g[noms[3 * i + 2]]] for i in range(3)]
    return lignes


def afficher_desk(monnaie):
    """Va chercher vos variables par leur nom, et dessine la fiche avec tout ce qui existe déjà."""
    g = globals()
    lire = lambda nom: g.get(nom)
    def essayer(f):
        try:
            return f()
        except Exception:
            return None
    mv = _mesures("mae_bete", "rmse_bete", "r2_bete", "mae_pers", "rmse_pers", "r2_pers",
                  "mae_modele", "rmse_modele", "r2_modele")
    mh = _mesures("mae_bete_h", "rmse_bete_h", "r2_bete_h", "mae_pers_h", "rmse_pers_h", "r2_pers_h",
                  "mae_har", "rmse_har", "r2_har")
    r = lire("r")
    dessiner_desk(
        monnaie,
        periode=essayer(lambda: f"{len(r)} jours de rendements  ·  {r['date'].min():%d/%m/%Y} → {r['date'].max():%d/%m/%Y}"),
        r=r, beta=lire("beta"), p_beta=lire("p_beta"), nu=lire("y_test"), couvert=lire("couvert"),
        vol_nue=lire("vol_nue"), vol_couverte=lire("vol_couverte"), reduction=lire("reduction"),
        pire_nu=lire("pire_nu"), pire_couvert=lire("pire_couvert"),
        dates_v=essayer(lambda: g["donnees"]["date"][g["X_test_v"].index]), reel_v=lire("y_test_v"),
        prevu_v=lire("prev_modele"),
        mesures_v=None if mv is None else pd.DataFrame(mv, columns=["MAE", "RMSE", "R²"],
                                                        index=["modèle bête", "persistance", "régression"]),
        prevision_5j=lire("prevision_5j"),
        minutes_jour=lire("jour"), rv_10oct=lire("rv_10oct"), rendement_10oct=lire("rendement_10oct"),
        rv_normal=lire("rv_normal"),
        dates_h=essayer(lambda: g["donnees_h"]["date"][g["X_test_h"].index]), reel_h=lire("y_test_h"),
        prevu_h=lire("prev_har"),
        mesures_h=None if mh is None else pd.DataFrame(mh, columns=["MAE", "RMSE", "R²"],
                                                        index=["modèle bête", "persistance", "HAR"]),
        prevision_demain=lire("prevision_demain"), rapport=lire("rapport"), part_pire=lire("part_pire"),
        pire_date=lire("pire_date"))
    plt.show()

## Regardez ce qu'on va remplir

Choisissez votre monnaie parmi les six : `ETH`, `SOL`, `DOGE`, `XRP`, `BNB`, `ADA`. Le bitcoin sert de **marché** : c'est contre lui qu'on couvre, on ne peut donc pas le choisir.

Si vous hésitez, gardez `ETH`. Si vous aimez les surprises, prenez `ADA`, `DOGE` ou `XRP` : le troisième volet vous réserve une enquête.

In [ ]:
MONNAIE = "ETH"

In [ ]:
rendements = pd.read_csv(BASE_STATS + "rendements.csv")
rendements["date"] = pd.to_datetime(rendements["date"])
crypto = pd.read_csv(BASE + "crypto.csv")
crypto["date"] = pd.to_datetime(crypto["date"])
vola = pd.read_csv(BASE_ML + "volatilite.csv")
vola["date"] = pd.to_datetime(vola["date"])
minutes = pd.read_csv(BASE_ML + "minutes_10oct2025.csv")

verifier("une monnaie possible", MONNAIE in ["ETH", "SOL", "DOGE", "XRP", "BNB", "ADA"], "une des six, entre guillemets, en majuscules")
afficher_desk(MONNAIE)

Voilà le squelette, en trois volets — un par collègue.

- **En haut, la couverture** pour Karim : que devient 10 000 € quand on se couvre, et cinq chiffres.
- **Au milieu, la liquidité** pour Léa : le volume réel et prévu, et trois prévisions comparées.
- **En bas, la volatilité** pour Tom : un jour de krach vu minute par minute, et la prévision de demain.

Tout est gris. **À chaque étape, vous allez lui en donner un peu plus.**

---

# Premier volet — la couverture

> Dans ce volet, **on ne prévoit rien**. On mesure une relation, et on s'en sert. C'est la leçon de la séance 7, section 4 : expliquer n'est pas prévoir — mais expliquer, ça peut rapporter.

## Étape 1 — Deux colonnes de rendements

**Ce qu'on cherche.** Une table `r` : la table `rendements`, réduite à **trois colonnes** — `date`, `BTC` et votre monnaie — **sans aucune ligne incomplète**, et renumérotée à partir de 0.

**Où c'était.** Séance 7, section 1 : le chargement de `rendements`. Choisir des colonnes avec une liste entre doubles crochets, `dropna`, `reset_index(drop=True)` : bloc 2.

**Le piège.** Votre monnaie s'appelle `MONNAIE`, pas `"MONNAIE"` : c'est une variable, sans guillemets. Et SOL n'est coté qu'à partir d'avril 2020 : sans `dropna`, il y aurait des lignes vides, et `fit` refuserait de travailler.

In [ ]:
_r = rendements[["date", "BTC", MONNAIE]].dropna()
verifier("les trois colonnes", list(r.columns) == ["date", "BTC", MONNAIE], "rendements[['date', 'BTC', MONNAIE]]")
verifier("aucune ligne incomplète", r.isna().sum().sum() == 0 and len(r) == len(_r), "dropna() sur la table réduite")
verifier("renumérotée", r.index[0] == 0 and r.index[-1] == len(r) - 1, "reset_index(drop=True)")

## Étape 2 — Le bêta, appris sur le passé

**Ce qu'on cherche.**

- `X_train, X_test, y_train, y_test` : le découpage, avec `BTC` pour ce qu'on donne et votre monnaie pour ce qu'on cherche, un cinquième mis de côté ;
- `beta` : le coefficient de la régression, **appris sur l'entraînement seulement** ;
- `p_beta` : sa p-value.

**Où c'était.** Séance 7, section 1 pour le découpage, section 2 pour `fit` et `coef_`, section 4 pour `stats.linregress` et sa p-value.

**Les pièges.**

- **`shuffle=False`.** Sans lui, on apprendrait le bêta avec des jours postérieurs à ceux qu'on va tester.
- **Ce qu'on donne doit être une table** : `r[["BTC"]]`, deux crochets. Avec un seul, `fit` affiche *Expected 2D array*. Si vous voyez ce message, c'est ça.
- **`coef_` est une liste** d'un seul élément : il faut `[0]`.
- `stats.linregress` veut deux **colonnes** : `X_train["BTC"]`, un crochet cette fois.

In [ ]:
_a, _b, _c, _d = train_test_split(r[["BTC"]], r[MONNAIE], test_size=0.2, shuffle=False)
verifier("le découpage", len(X_train) == len(_a) and len(X_test) == len(_b), "train_test_split(..., test_size=0.2, shuffle=False)")
verifier("dans l'ordre des dates", X_test.index.min() > X_train.index.max(), "shuffle=False : on apprend sur le passé, on teste sur le futur")
verifier("beta", abs(beta - LinearRegression().fit(_a, _c).coef_[0]) < 1e-9, "LinearRegression().fit(X_train, y_train).coef_[0] — sur l'entraînement seulement")
verifier("p_beta", abs(p_beta - stats.linregress(_a["BTC"], _c).pvalue) < 1e-12, "stats.linregress(...) sur X_train['BTC'] et y_train, puis .pvalue")

**Lecture.** Sur ETH, **β = 1,05** : quand BTC fait 1 %, ETH fait 1,05 % le même jour. La p-value est indiscernable de zéro : ce n'est pas un hasard de l'échantillon.

In [ ]:
afficher_desk(MONNAIE)

## Étape 3 — La position couverte, jugée sur l'avenir

**L'idée, en une image.** Karim a 10 000 € de votre monnaie. Il **vend** en même temps β × 10 000 € de bitcoin. Quand le marché monte, il gagne sur l'une et perd sur l'autre ; quand il baisse, pareil. Ce qui reste, c'est **ce qui est propre à votre monnaie**.

Jour par jour, le rendement de la position couverte, c'est donc : **le rendement de votre monnaie, moins β fois celui du bitcoin.**

**Pourquoi le test.** β a été appris sur le passé. Karim veut savoir si la couverture **aurait marché** sur une période que β n'a jamais vue. C'est la question de tout le bloc.

**Ce qu'on cherche**, tout sur le **test** :

| variable | ce que c'est |
|---|---|
| `couvert` | le rendement quotidien de la position couverte |
| `vol_nue`, `vol_couverte` | les deux volatilités : l'écart-type (bloc 3) de `y_test` et de `couvert` |
| `pire_nu`, `pire_couvert` | les deux pires jours |
| `reduction` | la baisse de la volatilité, **en %** : 1 − couverte / nue, fois 100 |

**Où c'était.** L'arithmétique sur des colonnes (bloc 2), `std` et `min` (bloc 3).

In [ ]:
_cv = y_test - beta * X_test["BTC"]
verifier("couvert", len(couvert) == len(y_test) and abs(couvert - _cv).max() < 1e-9, "y_test - beta * X_test['BTC'] : on soustrait, on n'ajoute pas")
verifier("les volatilités", abs(vol_nue - y_test.std()) < 1e-9 and abs(vol_couverte - _cv.std()) < 1e-9, ".std() sur y_test, puis sur couvert")
verifier("la couverture réduit le risque", vol_couverte < vol_nue, "si c'est l'inverse, vous avez peut-être ajouté au lieu de soustraire")
verifier("les pires jours", abs(pire_nu - y_test.min()) < 1e-9 and abs(pire_couvert - _cv.min()) < 1e-9, ".min()")
verifier("reduction en %", abs(reduction - (1 - _cv.std() / y_test.std()) * 100) < 1e-6, "(1 - vol_couverte / vol_nue) * 100 : un nombre entre 0 et 100")

In [ ]:
afficher_desk(MONNAIE)

**Lecture.** Sur ETH : volatilité **3,70 % → 2,10 %** par jour, soit **−43 %** de risque ; pire jour **−15,0 % → −6,1 %**. Un chiffre appris sur 2018-2024, appliqué à une période jamais vue, retire presque la moitié du risque d'une vraie position.

Regardez la courbe : la position couverte est **plus calme**, mais pas forcément plus haute. On y revient dans la question 2.

### Deux questions pour Karim

**1.** Refaites le calcul avec β = 1 au lieu de votre β : `reduction_beta1`. Que perdez-vous ?

In [ ]:
verifier("reduction_beta1", abs(reduction_beta1 - (1 - (y_test - X_test["BTC"]).std() / y_test.std()) * 100) < 1e-6, "la même formule, avec X_test['BTC'] au lieu de beta * X_test['BTC']")

**Que perdez-vous en prenant β = 1 ?** Et la couverture a-t-elle un prix ? *(Indice : si le bitcoin double demain, que gagne Karim ?)*

*[votre réponse]*

---

# Second volet — la liquidité

> Cette fois, **on prévoit**. La question n'est plus « comment ça bouge ensemble ? », c'est « combien, dans les jours qui viennent ? ». C'est la séance 7, section 6, sur une autre monnaie et un autre horizon.

## Étape 4 — Le volume, et la moyenne des cinq prochains jours

Léa ne vend pas en un jour : elle étale sur une semaine. Elle ne veut donc pas le volume de **demain**, mais la **moyenne des cinq prochains jours**. Il faut un geste nouveau. Il assemble deux gestes connus.

### Le geste nouveau, en trois temps

1. `rolling(5).mean()` donne, sur chaque ligne, la moyenne des 5 **derniers** jours, aujourd'hui compris.
2. `shift(-5)` remonte cette colonne de 5 lignes. Sur la ligne du lundi, on lit donc ce qui était calculé sur la ligne du samedi suivant : la moyenne de mardi à samedi. **Les 5 jours qui suivent.**
3. On vérifie avec ses yeux, avant de faire confiance.

In [ ]:
essai = pd.DataFrame({"jour": ["lun", "mar", "mer", "jeu", "ven", "sam", "dim", "lun"],
                      "vol": [10.0, 20.0, 30.0, 40.0, 50.0, 60.0, 70.0, 80.0]})
essai["moyenne des 5 derniers"] = essai["vol"].rolling(5).mean()
essai["moyenne des 5 suivants"] = essai["vol"].rolling(5).mean().shift(-5)
essai

Sur la ligne du premier lundi : la moyenne des 5 suivants vaut (20 + 30 + 40 + 50 + 60) / 5 = **40**. C'est juste. Les dernières lignes n'ont pas cinq jours devant elles : `NaN`.

**Ce qu'on cherche.** Une table `v` : les lignes de votre monnaie dans `crypto`, triées par date et renumérotées, avec quatre colonnes nouvelles.

| colonne | ce que c'est |
|---|---|
| `vol` | le montant échangé, en **milliards de dollars** : volume × close / 1e9 |
| `m5`, `m20` | ses moyennes sur les 5 et les 20 derniers jours |
| `cible` | la moyenne des **5 prochains** jours |

Puis `donnees` : `v` sans les lignes où `m20` ou `cible` manquent.

**Où c'était.** Séance 7, section 6, pour `vol`, `m5` et `m20`. Le geste du dessus pour `cible`.

**Le piège.** `shift(5)` au lieu de `shift(-5)` : on « prévoirait » la semaine **passée**, et le modèle serait excellent. Un R² trop beau pour être vrai veut souvent dire qu'on regarde en arrière.

In [ ]:
_v = crypto.query("coin == @MONNAIE").sort_values("date").reset_index(drop=True)
_v["vol"] = _v["volume"] * _v["close"] / 1e9
verifier("vol en milliards", abs(v["vol"].median() - _v["vol"].median()) < 1e-6, "volume × close / 1e9 ; si vos nombres sont énormes, il manque la division")
verifier("m5 et m20", abs(v["m20"][40] - _v["vol"][21:41].mean()) < 1e-6 and abs(v["m5"][40] - _v["vol"][36:41].mean()) < 1e-6, "rolling(20).mean() et rolling(5).mean() sur vol")
verifier("la cible regarde devant", abs(v["cible"][100] - _v["vol"][101:106].mean()) < 1e-6, "rolling(5).mean().shift(-5) ; avec shift(5), on regarde en arrière")
verifier("donnees", len(donnees) == len(_v) - 19 - 5 and donnees["cible"].isna().sum() == 0, "dropna(subset=['m20', 'cible'])")

## Étape 5 — Trois prévisions, trois mesures

**Ce qu'on cherche, en deux cellules.**

**D'abord** le découpage — `X_train_v, X_test_v, y_train_v, y_test_v`, avec les trois colonnes `vol`, `m5`, `m20` pour ce qu'on donne, `cible` pour ce qu'on cherche — puis `modele_v`, la régression ajustée sur l'entraînement.

**Ensuite** trois prévisions sur le test, et les trois mesures de chacune :

| prévision | ce qu'elle dit | les trois mesures |
|---|---|---|
| `prev_bete` | toujours la moyenne de l'entraînement : `np.full(len(y_test_v), y_train_v.mean())` | `mae_bete`, `rmse_bete`, `r2_bete` |
| `prev_pers` | la **persistance** : « les 5 prochains jours ressembleront à aujourd'hui » — la colonne `vol` du test | `mae_pers`, `rmse_pers`, `r2_pers` |
| `prev_modele` | la régression : `predict` | `mae_modele`, `rmse_modele`, `r2_modele` |

Neuf lignes de mesures, trois fois les mêmes : **copiez, collez, changez les noms**.

**Où c'était.** Séance 7, section 3 pour les mesures, section 6 pour la persistance et le modèle bête.

In [ ]:
_a, _b, _c, _d = train_test_split(donnees[["vol", "m5", "m20"]], donnees["cible"], test_size=0.2, shuffle=False)
verifier("le découpage", len(X_test_v) == len(_b) and list(X_train_v.columns) == ["vol", "m5", "m20"], "trois colonnes dans X : donnees[['vol', 'm5', 'm20']]")
verifier("modele_v", abs(modele_v.coef_ - LinearRegression().fit(_a, _c).coef_).max() < 1e-9, "fit sur X_train_v, y_train_v")

In [ ]:
_pm = modele_v.predict(X_test_v)
_pb = np.full(len(y_test_v), y_train_v.mean())
verifier("modèle bête", abs(mae_bete - mean_absolute_error(y_test_v, _pb)) < 1e-6 and abs(r2_bete - r2_score(y_test_v, _pb)) < 1e-6, "np.full(len(y_test_v), y_train_v.mean()) ; un R² négatif ici n'est pas une erreur")
verifier("persistance", abs(mae_pers - mean_absolute_error(y_test_v, X_test_v["vol"])) < 1e-6 and abs(rmse_pers - mean_squared_error(y_test_v, X_test_v["vol"]) ** 0.5) < 1e-6, "la prévision de la persistance est X_test_v['vol']")
verifier("régression", abs(r2_modele - r2_score(y_test_v, _pm)) < 1e-6 and abs(rmse_modele - mean_squared_error(y_test_v, _pm) ** 0.5) < 1e-6, "prev_modele = modele_v.predict(X_test_v) ; le RMSE est la racine : ** 0.5")

Il reste la réponse à Léa : **la prévision pour les cinq jours qui suivent la dernière date des données**. C'est la dernière ligne de `v` — celle dont la cible est encore inconnue, puisque ces jours ne sont pas arrivés. `tail(1)` la prend, **sous forme de table** ; `predict` renvoie une liste d'un élément, d'où le `[0]`.

**Ce qu'on cherche.** `prevision_5j`.

In [ ]:
verifier("prevision_5j", abs(prevision_5j - modele_v.predict(v[["vol", "m5", "m20"]].tail(1))[0]) < 1e-9, "modele_v.predict(v[['vol', 'm5', 'm20']].tail(1))[0] — sur v, pas sur donnees")
afficher_desk(MONNAIE)

**Lecture.** Sur ETH, le R² passe de **−0,92** (modèle bête) à **+0,18** (persistance) puis **+0,58** (régression). En vert sur la fiche, le meilleur de chaque colonne : c'est la régression, sur les trois mesures, et c'est vrai pour les six monnaies.

La persistance, si forte à un jour (séance 7), faiblit à cinq : un jour exceptionnel ne dure pas une semaine. La régression tient, parce qu'elle a appris à **peser** aujourd'hui contre la semaine et le mois.

## Étape 6 — Entraînement contre test

**Ce qu'on cherche.** Les mesures de la régression **sur l'entraînement** : `mae_train` et `r2_train`. Puis l'erreur rapportée au volume moyen de chaque période, en % : `mae_relative_train` (la MAE de l'entraînement divisée par la moyenne de `y_train_v`, fois 100) et `mae_relative_test` (pareil sur le test, avec `mae_modele`).

**Où c'était.** Séance 7, section 7.

In [ ]:
_q = modele_v.predict(X_train_v)
verifier("mae_train et r2_train", abs(mae_train - mean_absolute_error(y_train_v, _q)) < 1e-6 and abs(r2_train - r2_score(y_train_v, _q)) < 1e-6, "les prévisions sur X_train_v, comparées à y_train_v")
verifier("mae_relative_train", abs(mae_relative_train - mean_absolute_error(y_train_v, _q) / y_train_v.mean() * 100) < 1e-6, "mae_train / y_train_v.mean() * 100")
verifier("mae_relative_test", abs(mae_relative_test - mae_modele / y_test_v.mean() * 100) < 1e-6, "mae_modele / y_test_v.mean() * 100")

**Le R² est-il meilleur sur l'entraînement ? Et la MAE en % ?** Qu'est-ce que ça vous apprend sur la façon de comparer deux périodes ?

*[votre réponse]*

---

# Troisième volet — la volatilité

> Tom ne demande pas si ça va monter. Il demande **à quel point ça va bouger**. Ce n'est pas la même question — et ce n'est pas la même réponse.

## Étape 7 — Qu'est-ce que l'agitation d'une journée ?

**Le problème.** Le rendement du jour ne dit pas si la journée a été agitée. Un cours peut perdre 10 % à 14 h, tout reprendre à 18 h, et finir à 0 %. Pour mesurer l'agitation, il faut regarder **dans** la journée.

**La volatilité réalisée.** On prend les 1 440 prix de la journée, minute par minute. On calcule les rendements d'une minute à l'autre, on les élève au carré, on les additionne, et on prend la racine. **Beaucoup de grands mouvements, dans un sens ou dans l'autre, donnent un grand nombre.** C'est l'idée de l'écart-type du bloc 3, appliquée aux minutes d'une seule journée.

On va la calculer pour **le 10 octobre 2025**, un jour de krach éclair sur toutes les cryptos. `minutes` contient les 1 440 prix de chaque monnaie ce jour-là.

**Ce qu'on cherche.**

| variable | ce que c'est |
|---|---|
| `jour` | les lignes de votre monnaie dans `minutes` |
| `rv_10oct` | la volatilité réalisée de la journée, en % |
| `rendement_10oct` | le rendement de toute la journée, en % : dernier prix / premier prix − 1, fois 100 |
| `rv_normal` | la volatilité réalisée d'une journée **ordinaire** : la médiane de la colonne `rv` de `vola`, pour votre monnaie |

**Où c'était.** `pct_change() * 100` (bloc 3), les carrés et la somme (bloc 1), `np.sqrt` (bloc 1), la médiane (bloc 3).

**Un geste de plus.** Le premier et le dernier prix d'une colonne : `jour["close"].iloc[0]` et `jour["close"].iloc[-1]`. `iloc` compte les positions, comme les index d'une liste au bloc 1 — y compris le `-1` pour le dernier.

In [ ]:
_ref = vola[(vola["coin"] == MONNAIE) & (vola["date"] == pd.Timestamp("2025-10-10"))]["rv"].iloc[0]
verifier("jour", len(jour) == 1440 and jour["coin"].nunique() == 1, "minutes.query('coin == @MONNAIE')")
verifier("rv_10oct", abs(rv_10oct - _ref) < 0.01, "la racine (np.sqrt) de la somme des carrés des rendements minute, pct_change() * 100")
verifier("rendement_10oct", abs(rendement_10oct - (jour["close"].iloc[-1] / jour["close"].iloc[0] - 1) * 100) < 1e-9, "(dernier / premier - 1) * 100, avec iloc[-1] et iloc[0]")
verifier("rv_normal", abs(rv_normal - vola.query("coin == @MONNAIE")["rv"].median()) < 1e-9, "la médiane de la colonne rv, pour votre monnaie")
afficher_desk(MONNAIE)

**Lecture.** Regardez le petit graphique : la journée entière, minute par minute. Sur ETH, la volatilité réalisée est de **11,4 %**, soit **quatre fois** une journée ordinaire (2,9 %).

Et comparez à la valeur absolue du rendement de la journée. Sur ADA, DOGE ou XRP, la volatilité réalisée (entre 38 et 48 %) fait **le double** du rendement (−15 à −22 %) : le cours est descendu bien plus bas que son niveau de clôture, avant de remonter. Le rendement de la journée, à lui seul, n'aurait rien dit de ce vertige.

`vola`, que vous avez chargée au début, contient ce nombre pour chaque jour depuis 2022, calculé **exactement** comme vous venez de le faire — le `verifier` l'a confirmé.

## Étape 8 — Le modèle HAR : la même recette que le volume

Le modèle de prévision de la volatilité le plus utilisé, dans la recherche comme dans les banques, s'appelle **HAR** (Corsi, 2009). C'est une **régression linéaire** à trois variables : la volatilité d'aujourd'hui, sa moyenne sur 5 jours, sa moyenne sur 22 jours. L'idée derrière : trois sortes d'intervenants sur le marché — ceux qui traitent à la journée, à la semaine, au mois — et chacun laisse sa trace.

**C'est exactement le modèle du volume.** Vous savez déjà l'écrire.

**Ce qu'on cherche, en deux cellules.**

**D'abord** la table `h` : les lignes de votre monnaie dans `vola`, renumérotées, avec `rv5`, `rv22` (moyennes sur 5 et 22 jours) et `cible`, la volatilité de **demain**. Puis `donnees_h`, sans les lignes où `rv22` ou `cible` manquent. Puis le découpage `X_train_h, X_test_h, y_train_h, y_test_h` et `modele_h`.

**Ensuite** les trois prévisions — `prev_bete_h`, `prev_pers_h` (« demain = aujourd'hui », la colonne `rv` du test), `prev_har` — et leurs neuf mesures : `mae_bete_h`, `rmse_bete_h`, `r2_bete_h`, `mae_pers_h`, …, `mae_har`, `rmse_har`, `r2_har`. Et `prevision_demain`, comme `prevision_5j`.

**Où c'était.** Les étapes 4 et 5, en changeant les noms. Pour la cible de demain, `shift(-1)` (séance 7, section 5).

In [ ]:
_h = vola.query("coin == @MONNAIE").reset_index(drop=True)
verifier("rv5 et rv22", abs(h["rv22"][30] - _h["rv"][9:31].mean()) < 1e-9 and abs(h["rv5"][30] - _h["rv"][26:31].mean()) < 1e-9, "rolling(5).mean() et rolling(22).mean() sur rv")
verifier("la cible, c'est demain", abs(h["cible"][30] - _h["rv"][31]) < 1e-9, "shift(-1) sur rv")
_a, _b, _c, _d = train_test_split(donnees_h[["rv", "rv5", "rv22"]], donnees_h["cible"], test_size=0.2, shuffle=False)
verifier("le découpage", len(X_test_h) == len(_b) and list(X_test_h.columns) == ["rv", "rv5", "rv22"], "trois colonnes rv, rv5, rv22 ; test_size=0.2, shuffle=False")
verifier("modele_h", abs(modele_h.coef_ - LinearRegression().fit(_a, _c).coef_).max() < 1e-9, "fit sur X_train_h, y_train_h")

In [ ]:
_ph = modele_h.predict(X_test_h)
verifier("prev_har", len(prev_har) == len(y_test_h) and abs(prev_har - _ph).max() < 1e-9, "modele_h.predict(X_test_h)")
verifier("les mesures du HAR", abs(mae_har - mean_absolute_error(y_test_h, _ph)) < 1e-6 and abs(rmse_har - mean_squared_error(y_test_h, _ph) ** 0.5) < 1e-6 and abs(r2_har - r2_score(y_test_h, _ph)) < 1e-6, "les trois mesures, sur y_test_h et prev_har")
verifier("persistance et modèle bête", abs(r2_pers_h - r2_score(y_test_h, X_test_h["rv"])) < 1e-6 and abs(mae_bete_h - mean_absolute_error(y_test_h, np.full(len(y_test_h), y_train_h.mean()))) < 1e-6, "X_test_h['rv'], et la moyenne de y_train_h")
verifier("prevision_demain", abs(prevision_demain - modele_h.predict(h[["rv", "rv5", "rv22"]].tail(1))[0]) < 1e-9, "tail(1) sur h, comme à l'étape 5")
afficher_desk(MONNAIE)

**Lecture, sur ETH.** R² : **−0,01** (modèle bête), **+0,27** (persistance), **+0,42** (HAR).

> **La phrase de ce travail.** À la séance 7, le **rendement** de demain donnait un R² de zéro. Même monnaie, même méthode : l'**agitation** de demain en donne 0,42. *On ne sait pas dans quel sens le marché va bouger. On sait s'il va beaucoup bouger.* C'est ce que Tom fait chaque soir.

**Et si votre tableau est décevant ?** Sur ADA, DOGE ou XRP, le R² du HAR est proche de zéro, voire négatif, et la persistance fait pire que la moyenne. Ce n'est pas une erreur de votre part. C'est l'enquête de l'étape suivante.

## Étape 9 — L'enquête : la journée qui casse le modèle

Rappel de la séance 7, section 3 : si toutes les erreurs se valaient, le RMSE serait égal à la MAE. **Plus le rapport RMSE / MAE monte, plus quelques journées dominent l'erreur.** Autour de 1,4 : rien d'anormal. Au-dessus de 2 : il y a un coupable.

**Ce qu'on cherche.**

| variable | ce que c'est |
|---|---|
| `rapport` | le RMSE du HAR divisé par sa MAE |
| `erreurs` | l'écart absolu entre la réalité et la prévision du HAR, jour par jour du test |
| `pire_date` | la date de la ligne où l'erreur est la plus grande |
| `part_pire` | la part de la somme des erreurs au carré due à cette seule journée, en % |

**Où c'était.** `abs()` (bloc 2). `idxmax` donne l'étiquette de la ligne du maximum — c'est le `idxmin` du travail en autonomie du bloc 3. Avec cette étiquette, `h["date"][...]` lit la date.

In [ ]:
_e = (y_test_h - modele_h.predict(X_test_h)).abs()
verifier("rapport", abs(rapport - rmse_har / mae_har) < 1e-9, "rmse_har / mae_har")
verifier("pire_date", pd.Timestamp(pire_date) == h["date"][_e.idxmax()], "erreurs.idxmax() donne la ligne ; h['date'][...] donne la date")
verifier("part_pire en %", abs(part_pire - _e.max() ** 2 / (_e ** 2).sum() * 100) < 1e-6, "le carré de la plus grande erreur, divisé par la somme des carrés, fois 100")
afficher_desk(MONNAIE)

**Pour tout le monde, le coupable est le même** : la ligne du **9 octobre 2025**, celle qui devait prévoir le 10. Le modèle a annoncé une journée ordinaire. Le 10 octobre a été le krach éclair que vous avez regardé minute par minute à l'étape 7.

Sur ETH, cette journée pèse 19 % de l'erreur au carré. Sur ADA, **70 %** : une seule journée sur 337.

### Et sans elle ?

La cellule suivante met les jours du test dans une table, avec la prévision du HAR, et retire le krach : le 9 octobre (qui le prévoyait), le 10 (dont la volatilité folle sert de variable pour prévoir le 11) et le 11.

In [ ]:
test_h = donnees_h.tail(len(y_test_h)).copy()
test_h["prev_har"] = prev_har
hors_krach = test_h.query("date < '2025-10-09' or date > '2025-10-11'")
print(len(test_h), len(hors_krach))

**Ce qu'on cherche.** `r2_har_sans` et `r2_pers_sans` : les R² du HAR et de la persistance sur `hors_krach`. La réalité est la colonne `cible`, la prévision du HAR `prev_har`, celle de la persistance `rv`.

In [ ]:
verifier("r2_har_sans", abs(r2_har_sans - r2_score(hors_krach["cible"], hors_krach["prev_har"])) < 1e-9, "r2_score(hors_krach['cible'], hors_krach['prev_har'])")
verifier("r2_pers_sans", abs(r2_pers_sans - r2_score(hors_krach["cible"], hors_krach["rv"])) < 1e-9, "la persistance prévoit rv")
verifier("le HAR bat la persistance hors krach", r2_har_sans > r2_pers_sans, "relisez les deux lignes du dessus")

| | RMSE / MAE | poids du pire jour | R² du HAR | R² du HAR sans le krach | R² persistance sans le krach |
|---|---|---|---|---|---|
| ETH | 1,53 | 19 % | +0,42 | +0,49 | +0,38 |
| SOL | 1,41 | 14 % | +0,40 | +0,43 | +0,37 |
| BNB | 2,06 | 54 % | +0,37 | +0,57 | +0,50 |
| XRP | 2,62 | 66 % | +0,10 | +0,49 | +0,44 |
| DOGE | 2,62 | 62 % | −0,03 | +0,33 | +0,19 |
| ADA | **3,00** | **70 %** | **−0,07** | **+0,35** | +0,22 |

Hors du krach, le HAR bat la persistance **pour les six monnaies**.

**Le modèle est-il mauvais ?** Laquelle des trois mesures vous a menti, et laquelle vous a mis sur la piste ? Et que dites-vous à Tom ?

*[votre réponse]*

---

# Pour finir

## La fiche, pour n'importe quelle monnaie

Vos lignes, rassemblées dans une fonction : elle refait **tout** le notebook pour la monnaie qu'on lui donne. Elle est fournie ; relisez-la en diagonale, vous reconnaîtrez chaque étape.

In [ ]:
def fiche_de(nom):
    """Refait toutes les étapes de ce notebook pour la monnaie demandée."""
    rr = rendements[["date", "BTC", nom]].dropna().reset_index(drop=True)
    a, b, cc, d = train_test_split(rr[["BTC"]], rr[nom], test_size=0.2, shuffle=False)
    be = LinearRegression().fit(a, cc).coef_[0]
    cv = d - be * b["BTC"]

    vv = crypto.query("coin == @nom").sort_values("date").reset_index(drop=True)
    vv["vol"] = vv["volume"] * vv["close"] / 1e9
    vv["m5"] = vv["vol"].rolling(5).mean()
    vv["m20"] = vv["vol"].rolling(20).mean()
    vv["cible"] = vv["vol"].rolling(5).mean().shift(-5)
    dv = vv.dropna(subset=["m20", "cible"])
    a2, b2, c2, d2 = train_test_split(dv[["vol", "m5", "m20"]], dv["cible"], test_size=0.2, shuffle=False)
    mv = LinearRegression().fit(a2, c2)

    jj = minutes.query("coin == @nom")
    hh = vola.query("coin == @nom").reset_index(drop=True)
    hh["rv5"] = hh["rv"].rolling(5).mean()
    hh["rv22"] = hh["rv"].rolling(22).mean()
    hh["cible"] = hh["rv"].shift(-1)
    dh = hh.dropna(subset=["rv22", "cible"])
    a3, b3, c3, d3 = train_test_split(dh[["rv", "rv5", "rv22"]], dh["cible"], test_size=0.2, shuffle=False)
    mh = LinearRegression().fit(a3, c3)
    ph = mh.predict(b3)
    er = (d3 - ph).abs()

    def trois(y, p):
        return [mean_absolute_error(y, p), mean_squared_error(y, p) ** 0.5, r2_score(y, p)]

    dessiner_desk(
        nom, periode=f"{len(rr)} jours de rendements  ·  {rr['date'].min():%d/%m/%Y} → {rr['date'].max():%d/%m/%Y}",
        r=rr, beta=be, p_beta=stats.linregress(a["BTC"], cc).pvalue, nu=d, couvert=cv,
        vol_nue=d.std(), vol_couverte=cv.std(), reduction=(1 - cv.std() / d.std()) * 100,
        pire_nu=d.min(), pire_couvert=cv.min(),
        dates_v=dv["date"][b2.index], reel_v=d2, prevu_v=mv.predict(b2),
        mesures_v=pd.DataFrame([trois(d2, np.full(len(d2), c2.mean())), trois(d2, b2["vol"]), trois(d2, mv.predict(b2))],
                               columns=["MAE", "RMSE", "R²"], index=["modèle bête", "persistance", "régression"]),
        prevision_5j=mv.predict(vv[["vol", "m5", "m20"]].tail(1))[0],
        minutes_jour=jj, rv_10oct=np.sqrt(((jj["close"].pct_change() * 100) ** 2).sum()),
        rendement_10oct=(jj["close"].iloc[-1] / jj["close"].iloc[0] - 1) * 100, rv_normal=hh["rv"].median(),
        dates_h=dh["date"][b3.index], reel_h=d3, prevu_h=ph,
        mesures_h=pd.DataFrame([trois(d3, np.full(len(d3), c3.mean())), trois(d3, b3["rv"]), trois(d3, ph)],
                               columns=["MAE", "RMSE", "R²"], index=["modèle bête", "persistance", "HAR"]),
        prevision_demain=mh.predict(hh[["rv", "rv5", "rv22"]].tail(1))[0],
        rapport=trois(d3, ph)[1] / trois(d3, ph)[0], part_pire=er.max() ** 2 / (er ** 2).sum() * 100,
        pire_date=hh["date"][er.idxmax()])
    plt.show()

Essayez-la sur une autre monnaie que la vôtre — de préférence une qui raconte une autre histoire au troisième volet :

In [ ]:
fiche_de("ADA")

### Le sélecteur du desk

Dernière cellule, fournie elle aussi. Un menu déroulant : choisissez une monnaie, la fiche se recalcule. C'est toujours `fiche_de` qui travaille.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

menu = widgets.Dropdown(options=["ETH", "SOL", "DOGE", "XRP", "BNB", "ADA"], value=MONNAIE, description="Monnaie :")
sortie = widgets.Output()


def au_changement(_):
    with sortie:
        sortie.clear_output(wait=True)
        fiche_de(menu.value)


menu.observe(au_changement, names="value")
display(menu, sortie)
au_changement(None)

> Si cette dernière cellule n'affiche rien, ce n'est pas grave : le menu dépend d'un module que Colab ne charge pas toujours. `fiche_de("SOL")` marche dans tous les cas.

## Ce que vous en retenez

Répondez en double-cliquant sur les cellules « votre réponse ».

**1. Reprenez vos deux paris du début.** Quelle part du risque la couverture a-t-elle retirée ? Et lequel, du volume ou de l'agitation, s'est le mieux prévu ?

*[votre réponse]*

**2. Karim revient dans un an et utilise encore le même bêta. Qu'est-ce qui peut mal tourner ?**

*[votre réponse]*

**3. Léa, Karim ou Tom : qui repart avec l'outil le plus fiable, et pourquoi ?**

*[votre réponse]*

## La morale, en trois lignes

1. **Expliquer peut rapporter.** Le bêta ne prévoit rien, et il retire pourtant 25 à 45 % du risque.
2. **Ce qui a de la mémoire se prévoit** : le volume, l'agitation. Pas la direction du prix.
3. **Un modèle se juge aussi sur ses pires jours.** Le rapport RMSE / MAE l'a vu avant vous.

## Avant de fermer

1. *Exécution → Redémarrer et tout exécuter*.
2. Toutes les cellules `verifier` affichent `OK`.
3. Les trois réponses en texte sont écrites.